# QLoRA fine-tune

**Run this in Colab on a T4 (free) or an A100.** Nothing here works on a laptop: it needs a CUDA GPU
for 4-bit quantisation.

The plan: load a 3B base model in 4-bit, attach LoRA adapters to the attention projections, and train
on the tasting-note prompts so the model completes `Price is $` with a number. Only the adapters
train -- about 0.5% of the parameters -- which is what makes this fit in 16GB.

The library APIs here move fast, so the versions are floors rather than whatever Colab ships: `trl`
replaced its response-template collator with prompt-completion columns, and a stale cell fails at the
import.

Add two Colab secrets first (key icon, left sidebar): `HF_TOKEN` from
[huggingface.co](https://huggingface.co/settings/tokens) and `WANDB_API_KEY` from
[wandb.ai](https://wandb.ai/authorize). The run streams to Weights & Biases, which is how you watch a
four-hour fine-tune without leaving the tab open.

In [ ]:
!pip install -q "transformers>=4.56.2" "peft>=0.17" "trl>=1.0" "bitsandbytes>=0.44" \
    "datasets>=3.0" "accelerate>=1.0" "wandb>=0.18"
!git clone -q https://github.com/borjahernandez/wine-pricer.git
%cd wine-pricer

In [ ]:
import os
from datetime import datetime

import torch
import wandb
from datasets import load_dataset
from google.colab import userdata
from huggingface_hub import login
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

from pricer.prompts import as_completion

login(userdata.get("HF_TOKEN"))

BASE_MODEL = "Qwen/Qwen2.5-3B"
DATASET = "borjahernandez/wine-pricer"
RUN = "wine-pricer-qwen3b"
RUN_NAME = f"{RUN}-{datetime.now():%Y%m%d-%H%M}"  # one W&B run per attempt, so sweeps stay legible

os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
os.environ["WANDB_PROJECT"] = "wine-pricer"
os.environ["WANDB_LOG_MODEL"] = "false"  # adapters go to the Hub; W&B only needs the curves
os.environ["WANDB_WATCH"] = "false"  # gradient histograms cost throughput and rarely answer anything
wandb.login()

### Hyperparameters

Sensible starting points, all worth a sweep:

| knob | value | why |
| --- | --- | --- |
| `r` | 32 | adapter rank. 8 underfits here, 64 costs memory for little gain |
| `alpha` | 64 | conventionally 2r |
| target modules | attention projections | where the task-specific reasoning lives |
| `lr` | 1e-4 | LoRA tolerates rates ~10x a full fine-tune |
| epochs | 1 | 80k examples is plenty; a second epoch mostly memorises |
| 4-bit nf4, double quant | on | the whole reason this fits on a T4 |

In [ ]:
LORA = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

QUANT = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

CONFIG = SFTConfig(
    output_dir=RUN,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # effective batch 16
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=0.03,  # a float under 1 is read as a fraction of the run, so it tracks the split size
    optim="paged_adamw_32bit",
    max_length=256,
    completion_only_loss=True,
    logging_steps=50,
    save_steps=500,
    save_total_limit=2,
    bf16=True,
    report_to="wandb",
    run_name=RUN_NAME,
    push_to_hub=True,
    hub_model_id=f"borjahernandez/{RUN}",
    hub_private_repo=True,
)

In [ ]:
# The Hub dataset carries every curated field; the fine-tune reads one column pair. Splitting
# `prompt` at `Price is $` leaves the question as `prompt` and the bare price as `completion`, which is
# what `completion_only_loss` masks against -- the model is scored on the number, never on the prose.
data = load_dataset(DATASET)
train = data["train"].map(as_completion, input_columns="prompt", remove_columns=data["train"].column_names)
print(train)
print(train[0])

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=QUANT, device_map="auto")
model.generation_config.pad_token_id = tokenizer.pad_token_id

trainer = SFTTrainer(
    model=model,
    train_dataset=train,
    peft_config=LORA,
    args=CONFIG,
)

# Labels are built at map time now, and a prompt longer than `max_length` is dropped rather than
# truncated -- silently, since there is no exception and no loss spike to notice. Long notes are
# written about expensive bottles, so any loss lands in the thin top bins the balancing protects.
dropped = len(train) - len(trainer.train_dataset)
assert not dropped, f"{dropped} rows exceeded max_length={CONFIG.max_length} and were dropped"

trainer.train()
trainer.push_to_hub(f"Fine-tuned on {DATASET}")
wandb.finish()  # without this the run stays live and the summary metrics never settle

### Reading the W&B run

Three charts earn their place. `train/learning_rate` is the cheapest sanity check there is -- the ramp
should last ~3% of the steps and then decay on a cosine, which confirms the warmup fraction resolved
against the real step count rather than being read as an absolute value. `train/loss` on a
completion-only objective starts far lower than a full-text fine-tune, because only a handful of
price tokens are scored per example; watch its *slope*, not its height, and expect it to flatten
long before the epoch ends. `train/grad_norm` spiking after warmup means the learning rate is too
high for this rank.

The loss is not comparable to the baseline ladder -- that is what RMSLE on the held-out test set is
for, below. Nor is it comparable across curations: change the cap and the training pool changes with
it, so record which dataset a run used before trusting two loss curves side by side.

### Score it on the same test split as everything else

Two ways to read the answer out:

1. **Generate** a few tokens and parse the number.
2. **Weighted average over the logits** of the first answer token -- the model's whole distribution
   instead of its argmax, which is measurably better calibrated for a numeric target.

Both go through `pricer.evaluator`, so the result drops straight onto the same leaderboard as the
classical baselines.

In [ ]:
import re

from pricer.evaluator import evaluate
from pricer.items import Wine

_, _, test = Wine.from_hub(DATASET)
model.eval()


def parse_price(text: str) -> float:
    match = re.search(r"[-+]?\d[\d,]*\.?\d*", text.replace("$", ""))
    return float(match.group().replace(",", "")) if match else 0.0


def specialist(wine: Wine) -> float:
    inputs = tokenizer(wine.test_prompt(), return_tensors="pt").to("cuda")
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=6, do_sample=False)
    completion = tokenizer.decode(output[0][inputs["input_ids"].shape[1] :])
    return parse_price(completion)


specialist.__name__ = "Fine-tuned Qwen2.5-3B"
evaluate(specialist, test, size=250)

In [ ]:
def weighted(wine: Wine, top: int = 8) -> float:
    """Expected price under the model's own distribution over the first answer token."""
    inputs = tokenizer(wine.test_prompt(), return_tensors="pt").to("cuda")
    with torch.no_grad():
        logits = model(**inputs).logits[0, -1]
    probabilities = torch.nn.functional.softmax(logits, dim=-1)
    values, indices = probabilities.topk(top)
    prices, weights = [], []
    for probability, index in zip(values.tolist(), indices.tolist(), strict=True):
        price = parse_price(tokenizer.decode(index))
        if price:
            prices.append(price)
            weights.append(probability)
    if not prices:
        return 0.0
    total = sum(weights)
    return sum(price * weight for price, weight in zip(prices, weights, strict=True)) / total


weighted.__name__ = "Fine-tuned Qwen2.5-3B (weighted)"
evaluate(weighted, test, size=250)

### Experiments

- **Base model, untrained** on the same prompts: the gap is what the fine-tune actually bought.
- **Rank sweep**: r = 8 / 32 / 64 at matched steps.
- **Summaries vs full notes** (`-summaries` dataset from the previous notebook).
- **Add `points` to the prompt** and watch the fine-tune coast -- the same leakage the baselines see.
- **Bigger base**: an 8B model in 4-bit still fits an A100. Does scale beat data curation here?